# Blocked Plagiarised Docs — Diagnostic

Reads `analytics_summary.parquet` and identifies which plagiarised documents were blocked by Gate 1 (retrieval top-1 score below threshold).  
Use this to decide which cached per-doc parquets to delete before re-running with a lower `--min-top1-score`.

In [ ]:
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("pipeline_results")
PER_DOC_DIR = RESULTS_DIR / "per_doc"

df = pd.read_parquet(RESULTS_DIR / "analytics_summary.parquet")
df["gate1_passed"] = df["gate1_passed"].fillna(False)
df["is_clean"] = df["is_clean"].fillna(False)

print(f"Total docs processed : {len(df)}")
print(f"  Plagiarised (GT)    : {(~df['is_clean']).sum()}")
print(f"  Clean (GT)          : {df['is_clean'].sum()}")
df.head(3)

## Gate 1 breakdown

In [ ]:
plagiarised = df[~df["is_clean"]].copy()
clean       = df[df["is_clean"]].copy()

plag_blocked = plagiarised[~plagiarised["gate1_passed"]]
plag_passed  = plagiarised[plagiarised["gate1_passed"]]
clean_blocked = clean[~clean["gate1_passed"]]
clean_passed  = clean[clean["gate1_passed"]]

print("=== Gate 1 summary ===")
print(f"Plagiarised blocked (MISSED — bad)  : {len(plag_blocked):>3}  ({100*len(plag_blocked)/max(len(plagiarised),1):.0f}%)")
print(f"Plagiarised passed  (good)          : {len(plag_passed):>3}  ({100*len(plag_passed)/max(len(plagiarised),1):.0f}%)")
print(f"Clean blocked       (correct)       : {len(clean_blocked):>3}  ({100*len(clean_blocked)/max(len(clean),1):.0f}%)")
print(f"Clean passed (risk of FP at LLM)    : {len(clean_passed):>3}  ({100*len(clean_passed)/max(len(clean),1):.0f}%)")

## Blocked plagiarised docs — the ones that need a re-run

In [ ]:
cols_show = ["suspicious_doc_id", "gt_spans", "retrieval_top1_score",
             "branch_lsa_score", "branch_esa_score", "branch_emb_score",
             "f1", "gate1_passed"]

display_cols = [c for c in cols_show if c in plag_blocked.columns]
plag_blocked_disp = plag_blocked[display_cols].sort_values("suspicious_doc_id").reset_index(drop=True)
print(f"\n{len(plag_blocked)} plagiarised docs blocked by Gate 1 (all have F1=0.0):")
plag_blocked_disp

## Check which cached per-doc parquets exist for blocked docs

In [ ]:
rows = []
for doc_id in plag_blocked["suspicious_doc_id"]:
    cache_file = PER_DOC_DIR / f"{doc_id}.parquet"
    rows.append({
        "doc_id": doc_id,
        "gt_spans": int(plagiarised.loc[plagiarised["suspicious_doc_id"] == doc_id, "gt_spans"].iloc[0]),
        "cache_exists": cache_file.exists(),
        "cache_path": str(cache_file),
    })

cache_df = pd.DataFrame(rows)
print(f"Cache files found   : {cache_df['cache_exists'].sum()}")
print(f"Cache files missing : {(~cache_df['cache_exists']).sum()}")
cache_df

## Delete stale cache files and re-run commands

Run the cell below to **print** (not execute) the delete commands and the re-run command.  
Copy-paste into your terminal when ready.

In [ ]:
to_delete = cache_df[cache_df["cache_exists"]]

print("# --- PowerShell: delete stale cache files ---")
for _, row in to_delete.iterrows():
    print(f'Remove-Item "{row["cache_path"]}"')

print()
print("# --- then re-run the pipeline ---")
n_processed = len(df)
print(f"python scripts/final/run_pipeline.py --docs {n_processed} --skip-tfidf --run-embeddings --relative-gap 0.85 --min-top1-score 0.40")

## Passed plagiarised docs — performance check

In [ ]:
cols_pass = ["suspicious_doc_id", "gt_spans", "det_spans", "tp", "fp", "fn",
             "f1", "char_f1", "retrieval_top1_score", "branch_emb_score"]
display_cols_pass = [c for c in cols_pass if c in plag_passed.columns]

plag_passed_disp = plag_passed[display_cols_pass].sort_values("f1", ascending=False).reset_index(drop=True)
print(f"{len(plag_passed)} plagiarised docs that passed Gate 1:")
print(f"  Mean F1     : {plag_passed['f1'].mean():.3f}")
print(f"  Mean charF1 : {plag_passed['char_f1'].mean():.3f}")
print(f"  Perfect F1=1: {(plag_passed['f1']==1.0).sum()}")
print(f"  F1=0 (missed after Gate 1): {(plag_passed['f1']==0.0).sum()}")
plag_passed_disp

## Clean docs that passed Gate 1 — false alarm risk

In [ ]:
if len(clean_passed) == 0:
    print("No clean docs passed Gate 1 — zero false alarm risk from retrieval stage.")
else:
    cols_fp = ["suspicious_doc_id", "det_spans", "fp", "f1", "retrieval_top1_score"]
    display_cols_fp = [c for c in cols_fp if c in clean_passed.columns]
    fp_docs = clean_passed[clean_passed["fp"] > 0]
    print(f"{len(clean_passed)} clean docs passed Gate 1.")
    print(f"{len(fp_docs)} of those had false alarm detections (FP > 0).")
    clean_passed[display_cols_fp].sort_values("retrieval_top1_score", ascending=False).reset_index(drop=True)

## Score distribution: plagiarised vs clean (top-1 retrieval score)

In [ ]:
# Only docs that actually went through retrieval (gate1_passed=True or stale cache with real scores)
scored = df[df["retrieval_top1_score"] > 0].copy()
scored["label"] = scored["is_clean"].map({True: "clean", False: "plagiarised"})

print("Top-1 retrieval score stats (docs that went through retrieval):")
print(scored.groupby("label")["retrieval_top1_score"].describe().round(3).to_string())

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 4))
    for label, grp in scored.groupby("label"):
        ax.hist(grp["retrieval_top1_score"], bins=20, alpha=0.6, label=label)
    ax.axvline(0.40, color="red",    linestyle="--", label="Gate1=0.40")
    ax.axvline(0.70, color="orange", linestyle="--", label="Gate1=0.70 (old)")
    ax.set_xlabel("Top-1 retrieval score")
    ax.set_ylabel("# docs")
    ax.set_title("Score distribution — plagiarised vs clean")
    ax.legend()
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Plot skipped: {e}")